<a href="https://colab.research.google.com/github/IsuruKRanasundara/AI_Projects/blob/main/AIagent.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Install Required Libraries

In [2]:
!pip install transformers accelerate bitsandbytes
!pip install requests


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 13.9 MB/s eta 0:00:00


Load Mistral Model from Hugging Face

In [3]:
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch

model_name = "mistralai/Mistral-7B-Instruct-v0.2"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.float16,
    device_map="auto"
)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/596 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/493k [00:00<?, ?B/s]

...

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/111 [00:00<?, ?B/s]

Tool

In [4]:
import requests
from google.colab import userdata
def google_search(query):
    API_KEY =userdata.get("SERP")   # <-- Replace with your real key
    query = "AI News"

    url = "https://google.serper.dev/search"

    headers = {
        "X-API-KEY": API_KEY
    }

    params = {
        "q": query,
        "num": 3  # number of search results
    }

    response = requests.get(url, headers=headers, params=params)
    data = response.json()

    # Print top 3 results nicely
    if "organic" in data:
        for i, item in enumerate(data["organic"], 1):
            title = item.get("title", "")
            snippet = item.get("snippet", "")
            link = item.get("link", "")
            print(f"{i}. {title}\n{snippet}\n{link}\n")
    else:
        print("No results found.")

In [18]:
import json
import re

def execute_agent_tools(agent_response):
    """
    Safely execute the tool chosen by the AI agent.
    Extracts JSON from LLM output even if extra text is present.
    """

    try:
        # Use regex to find the first {...} block
        match = re.search(r'\{.*\}', agent_response, re.DOTALL)
        if not match:
            return "Error: Could not find JSON in agent response"

        data_json = match.group(0)
        data = json.loads(data_json)

        tool_name = data["tool"]
        input_value = data["input"]

        print("This is the tool name:", tool_name)
        print("Input to the tool:", input_value)

        # Call the corresponding tool
        if tool_name == "google_search":
            return google_search(input_value)
        elif tool_name == "get_temperature":
            return 'get_temperature(input_value)'
        elif tool_name == "word_count":
            return 'word_count(input_value)'
        else:
            return "Unknown tool"

    except json.JSONDecodeError:
        return "Error: Could not parse JSON from agent response"
    except KeyError:
        return "Error: JSON missing required keys"

In [38]:
import re

def ask_agent(user_query):
    # Prompt teaching the agent about tools
    system_prompt = f"""
You are an AI agent with access to the following tools:

1. google_search(query) → Returns search results from Google
2. get_temperature(city) → Returns current temperature of a city
3. word_count(text) → Returns the number of words in a text

Instructions:
- Decide which tool is needed based on the user's question.
- Respond ONLY in JSON format:
- ONLY provide one JSON object with the following format:
{{
  "tool": "tool_name",
  "input": "input_value"
}}
- Do not write anything else outside the JSON.
- Do not include system prompt in the JSON; only output the JSON object
"""

    full_prompt = f"{system_prompt}\nUser: {user_query}\nAssistant:"

    inputs = tokenizer(full_prompt, return_tensors="pt")

    outputs = model.generate(
        **inputs,
        max_new_tokens=200,
        do_sample=True,
        temperature=0.7,
        top_p=0.9,
        pad_token_id=tokenizer.eos_token_id  # avoid warnings
    )

    response = tokenizer.decode(outputs[0], skip_special_tokens=True)
    print("Raw LLM output:\n", response)

    # ---------------------------
    # Robust JSON extraction
    # ---------------------------
    match = re.search(r'\{.*?\}', response, re.DOTALL)
    if match:
        response_json = match.group(0)
    else:
        response_json = "{}"  # fallback empty JSON if none found

    return response_json

In [39]:
user_query = "Search for latest AI news"
agent_response = ask_agent(user_query)
print(agent_response)


# # Add opening brace if missing
# if not agent_response.strip().startswith("{"):
#     agent_response = "{" + agent_response

# result = execute_agent_tools(agent_response)
# print(result)
# agent_response = """
# Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
# {
#   "tool": "google_search",
#   "input": "latest AI news"
# }
# Extra text here
# """

result = execute_agent_tools(agent_response)
print(result)

Raw LLM output:
 
You are an AI agent with access to the following tools:

1. google_search(query) → Returns search results from Google
2. get_temperature(city) → Returns current temperature of a city
3. word_count(text) → Returns the number of words in a text

Instructions:
- Decide which tool is needed based on the user's question.
- Respond ONLY in JSON format:
- ONLY provide one JSON object with the following format:
{
  "tool": "tool_name",
  "input": "input_value"
}
- Do not write anything else outside the JSON.
- Do not include system prompt in the JSON; only output the JSON object

User: Search for latest AI news
Assistant: {"tool": "google_search", "input": "latest AI news"}
{
  "tool": "tool_name",
  "input": "input_value"
}
This is the tool name: tool_name
Input to the tool: input_value
Unknown tool
